# Estimation of Probabilistic Semi-Markovian Graphs

### *DGP (Data Generation Process)*

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="dowhy")   
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
def create_dataset(map_adjjj, N, number_of_obs, obs_nodes):
    map_datasett = {}
    for q in q_s:
        for p in p_s:
            nodes = []
            size = len(map_adjjj[(q,p)]["adj full matrix"][0])
            identity_matrix = np.eye(size)
            first_term = np.linalg.inv(identity_matrix-map_adjjj[(q,p)]["coefficients matrix"])
            full_data = np.empty((size,N))
            for i in range(N): 
                noise = np.random.normal(0, 1, (size, ))
        #print(f"current noise : {noise}\n")
                sample = first_term @ noise
        #print(f"current sample : {sample}\n")
                full_data[:, i] =  sample
            full_data = full_data.T
            for i in range(size-number_of_obs):
                nodes.append(f"U{i}")
            nodes.extend(obs_nodes)
                
            full_dataframe = pd.DataFrame(full_data, columns= nodes)
            observed_dataframe = full_dataframe.drop(columns=[col for col in full_dataframe if "U" in col], inplace=False)
            map_datasett[(q,p)] = {"full dataset": full_dataframe, "observed only":observed_dataframe}
    return map_datasett

Linear Model

In [ ]:
import numpy as np
import pandas as pd

# i can firstly define how many unobserved variables i have and only then construct matrix with corresponding size 

q_s = [0.2, 0.5, 0.7]
p_s = [0.2, 0.5, 0.7]
number_of_obs = 4 # # of observed nodes
obs_nodes = [ "X", "W", "V", "Z"]#[  "W", "V","X", "Z"]
seed = 40
np.random.seed(seed)
N = 7000

map_adj = {}
for q in q_s:
    for p in p_s:
        m = 0 # number of uncertain edges 

        #directed edges
        probs_for_bernoulli = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1) # just random floats under hauptdiagonale
        adj_directed = (probs_for_bernoulli >0) & (probs_for_bernoulli < q) # condition for existence of directed edge
        adj_directed = adj_directed.astype(int) # converting from T/F to 1/0
        probs = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1)
        probs = probs*adj_directed
        m+= len(np.argwhere((probs>0) & (probs<1)))
        
        #bidirected edges
        bidirected_edges = {}
        for i in range(number_of_obs):
            for j in range(number_of_obs):
                if i==j: 
                    continue
                    
                bi_prob = np.random.rand()
               
                if bi_prob>0 and bi_prob < p and (i, j) not in bidirected_edges and (j, i) not in bidirected_edges:
                    edge_prob = np.random.rand()
                    bidirected_edges[(i, j)] = edge_prob
                   
        n = len(bidirected_edges)#number of unobserved nodes
        m+= n # fix to be len(np.argwhere((bidirected_edges>0) & (bidirected_edges<1)))
        adj_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # initializing with appropriate size , matrix of 1/0
        prob_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # matrix of probs of edges
        #filling the bootom right block then take first bi_edge , take first column of full matrix, take row with index len(bidirected_edges)+i then len(bidirected_edges)+j and fill out 
        adj_full[n:, n:] = adj_directed
        prob_full[n:, n:] = probs
        for idx, (i,j) in enumerate(bidirected_edges.keys()):
            
            adj_full[n+i][idx] = 1 # skip all unobserved nodes (n+) , idx regulates move to the next unobserved
            adj_full[n+j][idx] = 1

            prob_full[n+i][idx]=bidirected_edges[(i,j)]
            prob_full[n+j][idx]=bidirected_edges[(i,j)]
            
        coef = np.tril(np.random.uniform(-1,1, (number_of_obs+n, number_of_obs+n)), k= -1) # lower triangular matrix (hauptdiagonale not included k<0), matrix of coefficients
        coef_matrix = adj_full*coef
        map_adj[(q,p)] = {"adj full matrix":adj_full, "coefficients matrix":coef_matrix, "probability matrix":prob_full, "number of uncertain edges": m, "number of unobserved":n}



map_dataset = create_dataset(map_adj, N, number_of_obs, obs_nodes)

1 Linear Model (Self-Defined Weights)

In [ ]:
map_adj_1 = {}

for q in q_s:
    for p in p_s:
        submatrix_directed = np.tril(np.random.binomial(1, q, size=(number_of_obs, number_of_obs)), k=-1)
        bidirected_edges = {}
        for i in range(number_of_obs):
            for j in range(number_of_obs):
                if i==j: 
                    continue
                    
                bi_prob = np.random.rand()
               
                if bi_prob>0 and bi_prob < p and (i, j) not in bidirected_edges and (j, i) not in bidirected_edges:
                    edge_prob = np.random.rand()
                    bidirected_edges[(i, j)] = 1
        n = len(bidirected_edges)#number of unobserved nodes
        adj_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # initializing with appropriate size , matrix of 1/0
        adj_full[n:, n:] = submatrix_directed
        for idx, (i,j) in enumerate(bidirected_edges.keys()):
            
            adj_full[n+i][idx] = 1 # skip all unobserved nodes (n+) , idx regulates move to the next unobserved
            adj_full[n+j][idx] = 1
            
        coef = np.tril(np.random.uniform(-1,1, (number_of_obs+n, number_of_obs+n)), k= -1) # lower triangular matrix (hauptdiagonale not included k<0), matrix of coefficients
        coef_matrix = adj_full*coef
        map_adj_1[(q,p)] = {"adj full matrix":adj_full, "coefficients matrix":coef_matrix, "number of unobserved":n}

map_dataset_1 = create_dataset(map_adj_1, N, number_of_obs, obs_nodes)

## Algorithm 1

### *Splitting Graph into Subgraphs*

In [ ]:
import scipy.stats as stats
from scipy.stats import differential_entropy
from sklearn.linear_model import LinearRegression

def weights (targetColumn, parentsColumns ):
    targetColumn = np.asarray(targetColumn).ravel()

    if parentsColumns.size==0:
        noise = targetColumn - targetColumn.mean()
    else:
        parentsColumns = np.asarray(parentsColumns)
        if parentsColumns.ndim ==1:
            parentsColumns = parentsColumns.reshape(-1,1)
        model = LinearRegression()
        model.fit( parentsColumns, targetColumn)
        noise = targetColumn - model.predict(parentsColumns)
    return float(differential_entropy(noise))

In [ ]:
#Linear Model
import networkx as nx
from itertools import product, islice

map_of_subgraphs = {} # (q,p) -> subgraohs -> subgraph + probability of its existence

for q in q_s:
    for p in p_s:
        uncertain_edges = np.argwhere((map_adj[(q,p)]["probability matrix"]>0) & (map_adj[(q,p)]["probability matrix"]<1))
        sorted_u_e = uncertain_edges[uncertain_edges[:, 1].argsort()] # sorting by columns so that by increasing index k of uncertain edge we move to next column
        m = map_adj[(q,p)]["number of uncertain edges"] #
        n = map_adj[(q,p)]["number of unobserved"] #
        subgraphs = []
        for combination in product([True, False], repeat=m): # 2^m combinations , m is the length of each combination, each element is either t or f
            subgraoh_prob = map_adj[(q,p)]["probability matrix"].copy()
            subgraph_adj = map_adj[(q,p)]["adj full matrix"].copy()
            prob_of_subgraph= 1
            k = 0
            for i,v in enumerate(combination):
                if v==False:
                    if i < n: # unobserved column -> delete both entries
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] # kth entry
                        subgraoh_prob[:, i] = 0
                        subgraph_adj[:,i] = 0
                        
                        k += 2 # deleted two subsequent entries
                    else:
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                        subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        subgraph_adj[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        k+=1
                        
                    
                else:
                    prob_of_subgraph *= subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                    if i < n:
                        k+=2
                    else: 
                        k+=1
            #calculating weight using entropy
            dataset = map_dataset[(q,p)]["observed only"]
            nodes = list(map_dataset[(q, p)]["full dataset"].columns)
            Z = nodes.index("Z")
            parents = subgraph_adj[Z].astype(bool)
            parent_names = [nodes[j] for j in np.where(parents)[0]]
            observed_parents = [p for p in parent_names if p in dataset.columns]
            Pa = dataset[observed_parents].to_numpy()
            H = weights(dataset["Z"].to_numpy(), Pa)
            weight = 1/ np.exp(H)
            
                 #, "weightEnt": weight         
            subgraphs.append({"combination":combination,"adj submatrix":subgraph_adj,"prob submatrix":subgraoh_prob, "prob of subgraph": prob_of_subgraph, "weightEnt": weight})
        map_of_subgraphs[(q,p)] = {"subgraphs":subgraphs}
    

## Algorithm 3 

### *Estimating Causal Effect of Subgraphs*

In [ ]:
import networkx as nx
from dowhy import CausalModel# can also use DOT file through parameter "graph"
                              # can also do the identification
from collections import defaultdict

import logging
logging.disable(logging.CRITICAL)
logging.getLogger("dowhy").disabled = True
x_s = list(range(-number_of_obs, number_of_obs+1))
map_of_estimates_and_weights = {}
for q in q_s:
    for p in p_s:
        dataset = map_dataset[(q,p)]["observed only"]
        meanZ = dataset["Z"].mean()
        meanX = dataset["X"].mean()
        estimates_and_weights = defaultdict(list)
        for subgraph in map_of_subgraphs[(q,p)]["subgraphs"]:
            graph = nx.from_numpy_array(subgraph["adj submatrix"].T, create_using=nx.DiGraph)
            graph = nx.relabel_nodes(graph,dict(enumerate(map_dataset[(q,p)]["full dataset"].columns.tolist())))
            model = CausalModel(data = map_dataset[(q,p)]["observed only"], treatment = "X", outcome = "Z", graph=graph )
            estimate = None
            
            try:
                identifiability = model.identify_effect(proceed_when_unidentifiable=False)
                estimands = identifiability.estimands
                ATE = None
                if estimands.get("backdoor"):
                    estimate = model.estimate_effect(identifiability, method_name="backdoor.linear_regression", treatment_value = 1)# compares to control value = 0, uses formula betaX * (treatment - control), since treatment = 1 and control = 0 -> we receive pure betaX 
                    ATE = estimate.value
                    
                elif estimands.get("frontdoor"):
                    estimate = model.estimate_effect(identifiability, method_name="frontdoor.two_stage_regression", treatment_value = 1)
                    ATE = estimate.value
                elif estimands.get("iv"):
                    estimate = model.estimate_effect(identifiability, method_name="iv.instrumental_variable", treatment_value = 1)
                    ATE = estimate.value
                if ATE is not None: 
                    for x in x_s:
                        estimated_target = ATE * x
                        #meanZ + ATE*(x-meanX)# if ATE is just slope -> ATE*x w/o other terms
                        estimates_and_weights[x].append({"estimate":estimated_target, "weight": subgraph["weightEnt"], "ATE": ATE})
                            
            except Exception as e:
                continue
        map_of_estimates_and_weights[(q,p)] = estimates_and_weights
            

In [ ]:
#Linear Model 1
map_of_estimates_and_weights_1 = {}
for q in q_s:
    for p in p_s:
        number_of_subgraphs = len(map_of_subgraphs_1[(q, p)]["subgraphs"])
        
        floats = np.random.rand(number_of_subgraphs)
        dataset = map_dataset_1[(q,p)]["observed only"]
        meanZ = dataset["Z"].mean()
        meanX = dataset["X"].mean()
        estimates_and_weights = defaultdict(list)
        
        for i,subgraph in enumerate(map_of_subgraphs_1[(q,p)]["subgraphs"]):
            graph = nx.from_numpy_array(subgraph["adj submatrix"].T, create_using=nx.DiGraph)
            graph = nx.relabel_nodes(graph,dict(enumerate(map_dataset_1[(q,p)]["full dataset"].columns.tolist())))
            model = CausalModel(data = map_dataset_1[(q,p)]["observed only"], treatment = "X", outcome = "Z", graph=graph )
            estimate = None
            
            try:
                identifiability = model.identify_effect(proceed_when_unidentifiable=False)
                estimands = identifiability.estimands
                ATE = None
                if estimands.get("backdoor"):
                    estimate = model.estimate_effect(identifiability, method_name="backdoor.linear_regression", treatment_value = 1)# compares to control value = 0, uses formula betaX * (treatment - control), since treatment = 1 and control = 0 -> we receive pure betaX 
                    ATE = estimate.value
                    
                    
                elif estimands.get("frontdoor"):
                    estimate = model.estimate_effect(identifiability, method_name="frontdoor.two_stage_regression", treatment_value = 1)
                    ATE = estimate.value
                    
                elif estimands.get("iv"):
                    estimate = model.estimate_effect(identifiability, method_name="iv.instrumental_variable", treatment_value = 1)
                    ATE = estimate.value
                    
                    
                if ATE is not None: 
                    for x in x_s:
                        estimated_target = ATE * x
                        estimates_and_weights[x].append({"estimate":estimated_target, "weight": subgraph["weightEnt"], "ATE": ATE})
                            
            except Exception as e:
                continue
        map_of_estimates_and_weights_1[(q,p)] = estimates_and_weights
    

### *Finding Wasserstein Barycenters of Estimates*

In [ ]:
import ot # another library for Wasserstein Barycenters
import numpy as np

def barycenters(map_of_estimates_and_weightsss):
    
    regularization = 5e-3

    n = 100 # number of bins
    y = np.linspace(-6, 6, n)# range of possible values of estimated causal effect
    
    
    M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean')# sqeuclidean -> squared wasserstein2 distance  
    M /= M.max()
    
    map_of_barycentersss = {}
    
    for q in q_s:
        for p in p_s:
            for x in x_s:
                distributions = []#among all subgraphs of a particular pair of q and p
                if map_of_estimates_and_weightsss[(q,p)][x] == []:
                    map_of_barycentersss[(q,p,x)] =[]
                    continue
                for e in map_of_estimates_and_weightsss[(q,p)][x]: # e is an estimate with corresponding weight#
                    mean = e["estimate"]
                    bin_index = np.argmin(np.abs(y-mean))# finds nearest bin to the estimate we want to make a mean 
                    distribution = ot.datasets.make_1D_gauss(n, m = bin_index, s=15)
                    distribution /= distribution.sum()
                    distributions.append(distribution)
                A = np.stack(distributions).T 
                weights = np.array([e["weight"] for e in map_of_estimates_and_weightsss[(q,p)][x]])
                weights /= weights.sum()
                barycenter = ot.bregman.barycenter(A,M, regularization, weights)# check only for gaussian
                barycenter /= barycenter.sum()
                map_of_barycentersss[(q,p,x)] = {"barycenter":barycenter, "A": A}
    return map_of_barycentersss
    


map_of_barycenters = barycenters(map_of_estimates_and_weights)  
map_of_barycenters_1 = barycenters(map_of_estimates_and_weights_1)
        


In [ ]:
      
        
import matplotlib.pyplot as plt
import numpy as np

def plotting(map_of_barycenters):
    qp_pairs = [(q, p) for q in q_s for p in p_s]
    n = 100 # number of bins
    y = np.linspace(-6, 6, n)# range of possible values of estimated causal effect

    num_x = len(x_s)
    num_cols = len(qp_pairs)
    num_rows = num_x * 2  # 2 rows for every x (one for A, one for Barycenter)
    
    
    fig, axes = plt.subplots(
        nrows=num_rows, 
        ncols=num_cols, 
        figsize=(num_cols * 5, num_rows * 3), 
        tight_layout=True
    )
    
    
    
    for i, x_val in enumerate(x_s):
        for j, (q, p) in enumerate(qp_pairs):
            
            row_top = i * 2
            row_bottom = i * 2 + 1
            
            ax1 = axes[row_top, j]
            ax2 = axes[row_bottom, j]
            
            
            if not map_of_barycenters.get((q, p, x_val)) or map_of_barycenters[(q, p, x_val)] == []:
                ax1.axis('off')
                ax2.axis('off')
                ax1.text(0.5, 0.5, f"Unidentifiable\nq={q}, p={p}, x={x_val}",  fontsize=20)
                continue
                
        
            ax1.plot(y, map_of_barycenters[(q, p, x_val)]["A"]) 
            ax1.set_title(f"Distributions q={q}, p={p} | do(X={x_val})")
         
            ax2.plot(y, map_of_barycenters[(q, p, x_val)]["barycenter"], color='orange')
            ax2.set_title(f"Barycenter | do(X={x_val})")
    
    plt.show()

plotting(map_of_barycenters)


In [ ]:
plotting(map_of_barycenters_1)

In [ ]:


def error_and_target(map_datasetttt,map_adjjjj, barycenters):
    map_of_errors = {}
    map_of_target = {}
    map_of_target_distr = {}
    for q in q_s:
        for p in p_s:
            data = map_datasetttt[(q,p)]["full dataset"]
            mean_z = data["Z"].mean()
            mean_x = data["X"].mean()
            estimates = defaultdict(list)
        
            graph = nx.from_numpy_array(map_adjjjj[(q,p)]["adj full matrix"].T, create_using=nx.DiGraph)
            graph = nx.relabel_nodes(graph, dict(enumerate(map_datasetttt[(q,p)]["full dataset"].columns.tolist())))
            
            
            model = CausalModel(data=data,treatment="X",outcome="Z",graph=graph)
            
            try:
                identifiability = model.identify_effect(proceed_when_unidentifiable=False)
                estimands = identifiability.estimands
                ATE = None
                if estimands.get("backdoor"):
                    #print(identifiability.backdoor_variables)
                    estimate = model.estimate_effect(identifiability, method_name="backdoor.linear_regression")
                    ATE = estimate.value
                    
                elif estimands.get("frontdoor"):
                    estimate = model.estimate_effect(identifiability, method_name="frontdoor.two_stage_regression")
                    ATE = estimate.value
                elif estimands.get("iv"):
                    estimate = model.estimate_effect(identifiability, method_name="iv.instrumental_variable")
                    ATE = estimate.value
                if ATE is not None: 
                    for x in x_s:
                        estimated_target = ATE*x
                        
                        estimates[x].append(estimated_target)
                            
            except Exception as e:
               
                continue
            
            map_of_target[(q,p)] = estimates
            
    regularization = 5e-3

    n = 100 # number of bins
    y = np.linspace(-6, 6, n)# range of possible values of estimated causal effect
    
    
    M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean')# sqeuclidean -> squared wasserstein2 distance  
    M /= M.max()

    for q in q_s:
        for p in p_s:
            for x in x_s:
                
                if (q,p) not in map_of_target.keys():
                    continue
                barycenter_entry = barycenters.get((q,p,x), [])
                if isinstance(barycenter_entry, list):
                    continue
                true_effect = map_of_target[(q,p)][x]
                bin_idx_te = np.argmin(np.abs(y-true_effect[0]))
                distribution_te = ot.datasets.make_1D_gauss(n, m = bin_idx_te, s=15)
                distribution_te /= distribution_te.sum() # otherwise size mismatch in emd2
                w2_error_2 = ot.emd2(distribution_te, barycenters[(q,p,x)]["barycenter"], M)
                w2_error = np.sqrt(w2_error_2)
                map_of_errors[(q,p,x)] = w2_error
                map_of_target_distr[(q,p,x)] = distribution_te
    return map_of_errors, map_of_target_distr



In [ ]:
map_of_errors, map_of_target_distr = error_and_target(map_dataset,map_adj, map_of_barycenters)
map_of_errors_1, map_of_target_distr_1 = error_and_target(map_dataset_1,map_adj_1, map_of_barycenters_1)

In [ ]:


import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np
import ot

def errors(map_of_barycenters, map_of_target_distr, map_of_errorssss):
    qp_pairs = [(q, p) for q in q_s for p in p_s] # flattened list of all (q, p)
    n = 100 # number of bins
    y = np.linspace(-6, 6, n) # range of possible values of estimated causal effect
    
    M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean') 
    M /= M.max()
    
    num_x = len(x_s)
    num_cols = len(qp_pairs)
    num_rows = len(x_s) # 1 row per x value
    
    # Grid initialization
    fig, axes = plt.subplots(
        nrows=num_rows, 
        ncols=num_cols, 
        figsize=(num_cols * 5, num_rows * 4), # to fit legend
        tight_layout=True
    )
    
    # Safeguard for reshaping
    if num_rows == 1 and num_cols == 1:
        axes = np.array([[axes]])
    else:
        axes = np.array(axes).reshape(num_rows, num_cols)
    
    for i, x_val in enumerate(x_s):
        for j, (q, p) in enumerate(qp_pairs):
            ax = axes[i, j] # one axis object per cell now
            bary_dict = map_of_barycenters.get((q, p, x_val))
            target_dist = map_of_target_distr.get((q, p, x_val))
            
            # Check for unidentifiable graphs or missing data
            if target_dist is None or not bary_dict or isinstance(bary_dict, list):
                # Hide the axes and write a clean label
                ax.axis('off')
                ax.text(0.5, 0.5, f"Unidentifiable\nq={q}, p={p}\nx={x_val}", 
                        ha='center', va='center', fontsize=20)
                continue
            
            # Extract matrices
            A_matrix = bary_dict["A"]
            barycenter = bary_dict["barycenter"]
            
            # 1. Plot Subgraph Distributions (Background Layer)
           
            ax.plot(y, A_matrix, color="gray", alpha=0.3)
            
            ax.plot([], [], color="gray", alpha=0.3, label="Subgraphs")
            
            # 2. Plot Barycenter (Middle Layer)
            ax.plot(y, barycenter, color="orange", label="Estimation", linewidth=2.5) 
            
            # 3. Plot Target Distribution (Top Layer)
            ax.plot(y, target_dist, color="blue", label="Target", linewidth=2.5, linestyle="--")
            
            ax.set_title(f"Distributions q={q}, p={p} | do(X={x_val}. Error: {map_of_errorssss[q,p,x_val]:.4f})")
            ax.legend(loc="upper right")
    
    plt.show()
    

errors(map_of_barycenters, map_of_target_distr, map_of_errors)

In [ ]:
errors(map_of_barycenters_1, map_of_target_distr_1, map_of_errors_1)